# Algoritma Pencarian Latin Square Komutatif atas Aljabar Max-Plus

## Langkah 1: Mencari Permutasi Elemen Terbesar
Langkah pertama adalah dari matriks awal $A$ (yang merupakan Latin Square), kita akan mencari permutasi yang posisinya menghasilkan nilai total yang maksimal (pada aljabar max-plus ini merepresentasikan elemen pada *cycle* terbesar).
Permutasi yang dihasilkan direpresentasikan menggunakan himpunan $\{1, 2, 3, \ldots, n\}$.

In [2]:
import numpy as np
from scipy.optimize import linear_sum_assignment
from sympy.combinatorics import Permutation

def get_max_weight_permutation(A):
    """
    Mencari permutasi elemen terbesar dengan algoritma eksak menggunakan pencocokan (linear_sum_assignment).
    Mengembalikan objek Permutation dari SymPy (0-based di dalam).
    """
    row_ind, col_ind = linear_sum_assignment(-A)
    return Permutation(list(col_ind))

def print_cycle_1_based(perm, n):
    """Standarisasi ke bentuk string sikel 1-based"""
    cycles = perm.cyclic_form
    # Jika permutasi identitas dan tidak ada sikel yang berpindah
    if not cycles:
        return "(1)"
    return "".join("(" + " ".join(str(i + 1) for i in c) + ")" for c in cycles)

# ================================
# Contoh Penggunaan: Matriks A berupa Latin Square (ukuran 3x3)
A = np.array([
    [1, 2, 3],
    [2, 3, 1],
    [3, 1, 2]
], dtype=int)

A = np.array([
    [4, 2, 3, 1],
    [1, 4, 2, 3],
    [3, 1, 4, 2],
    [2, 3, 1, 4]

], dtype=int)

pi_A = get_max_weight_permutation(A)

print("Matriks A (Latin Square):")
print(A)
print("\n1. Permutasi pusat matriks A (pi_A):")
print("   Array 1-based:", [pi_A(i) + 1 for i in range(len(A))])
print("   Notasi Sikel :", print_cycle_1_based(pi_A, len(A)))

Matriks A (Latin Square):
[[4 2 3 1]
 [1 4 2 3]
 [3 1 4 2]
 [2 3 1 4]]

1. Permutasi pusat matriks A (pi_A):
   Array 1-based: [1, 2, 3, 4]
   Notasi Sikel : (1)


## Langkah 2: Mencari Himpunan Centralizer

Setelah mendapatkan permutasi terbesarnya, kita mencari himpunan permutasi yang komutatif dengan `pi_A` (Centralizer-nya).

*Catatan package:* Karena SymPy memiliki *group theory computation* yang komprehensif lewat sub modul `combinatorics`, alih-alih melakukan *bruteforce* kita dapat menggunakan `SymmetricGroup().centralizer()` secara *native*  untuk men-generasi sikel yang selaras. Ini akan lebih efektif, ringkas, dan memunculkan representasi sikel secara instan.

In [3]:
from sympy.combinatorics import SymmetricGroup

def get_centralizer(pi, n):
    """
    Mencari centralizer dari permutasi pi pada grup simetri S_n.
    Mengembalikan daftar semua permutasi yang komutatif dengan pi
    berupa objek Permutation SymPy.
    """
    S_n = SymmetricGroup(n)
    # SymPy telah memiliki fungsi centralizer bawaan secara teori grup
    cent = S_n.centralizer(pi)
    
    # Hasilkan list elemen-elemennya yang masuk akal di grup simetrinya
    return list(cent.elements)

center_pi = get_centralizer(pi_A, len(A))

print("2. Himpunan Center dari pi_A (permutasi sigma yang komutatif):")
for p in center_pi:
    array_form = [p(i) + 1 for i in range(len(A))]
    cycle_str = print_cycle_1_based(p, len(A))
    print(f"   Array: {array_form} | Sikel: {cycle_str}")

2. Himpunan Center dari pi_A (permutasi sigma yang komutatif):
   Array: [1, 2, 3, 4] | Sikel: (1)
   Array: [2, 3, 4, 1] | Sikel: (1 2 3 4)
   Array: [3, 4, 1, 2] | Sikel: (1 3)(2 4)
   Array: [4, 2, 3, 1] | Sikel: (1 4)
   Array: [1, 3, 4, 2] | Sikel: (2 3 4)
   Array: [2, 4, 1, 3] | Sikel: (1 2 4 3)
   Array: [3, 1, 2, 4] | Sikel: (1 3 2)
   Array: [4, 3, 1, 2] | Sikel: (1 4 2 3)
   Array: [1, 4, 2, 3] | Sikel: (2 4 3)
   Array: [2, 1, 3, 4] | Sikel: (1 2)
   Array: [3, 2, 4, 1] | Sikel: (1 3 4)
   Array: [4, 1, 2, 3] | Sikel: (1 4 3 2)
   Array: [1, 2, 4, 3] | Sikel: (3 4)
   Array: [2, 3, 1, 4] | Sikel: (1 2 3)
   Array: [3, 4, 2, 1] | Sikel: (1 3 2 4)
   Array: [4, 2, 1, 3] | Sikel: (1 4 3)
   Array: [1, 3, 2, 4] | Sikel: (2 3)
   Array: [2, 4, 3, 1] | Sikel: (1 2 4)
   Array: [3, 1, 4, 2] | Sikel: (1 3 4 2)
   Array: [4, 3, 2, 1] | Sikel: (1 4)(2 3)
   Array: [1, 4, 3, 2] | Sikel: (2 4)
   Array: [2, 1, 4, 3] | Sikel: (1 2)(3 4)
   Array: [3, 2, 1, 4] | Sikel: (1 3)
   Array: [4

## Langkah 3: Membuat Matriks Sementara B

Karena matriks $B$ yang kita cari juga merupakan **Latin Square** berukuran $n \times n$, kita tahu bahwa elemen terbesarnya pasti bernilai $n$.
Agar komutatif secara aljabar Max-Plus, permutasi tempat nilai maksimum pada $B$ (sebut saja $\sigma$) harus diambil dari himpunan *Centralizer* $C(\pi_A)$ yang sudah kita temukan.

Matriks sementara $B$ akan dikonstruksi dengan cara mengisi nilai $n$ pada posisi yang bersesuaian dengan pemetaan $\sigma$, sedangkan sel matriks sisanya akan diberi nilai $-\infty$ (Identitas penjumlahan Max-plus). Hal ini menjadi pondasi (seed) matriks $B$ sebelum elemen yang lain dicari.

In [4]:
def get_B_temporary_matrix(sigma, n):
    """
    Membuat matriks sementara B berdasarkan pemetaan permutasi terbesarnya.
    Karena B adalah sebuah Latin Square berorde n, maka elemen terbesarnya bernilai n.
    Elemen pada posisi hasil pemetaan diubah ke n, sisanya di-set -inf.
    """
    B_Temp = np.full((n, n), -np.inf)
    max_val = float(n)  # Elemen maksimal pada Latin Square ordo n (dari nilai 1 s/d n)
    for i in range(n):
        # SymPy Permutation akan mengubah posisi sesuai mapping cycle zero-based
        j = sigma(i)  
        B_Temp[i, j] = max_val
    return B_Temp

# Mencoba matriks sementara berdasar semua elemen dari komutator (himpunan Center)
n_size = len(A)
B_temps = [get_B_temporary_matrix(p, n_size) for p in center_pi]

print(f"3. Kandidat Matriks Sementara B (Terdapat {len(B_temps)} kemungkinan letak elemen dominan n):")
for idx, (p, B_tmp) in enumerate(zip(center_pi, B_temps)):
    cycle_str = print_cycle_1_based(p, n_size)
    print(f"\nKandidat ke-{idx + 1} dari sikel: {cycle_str}")
    print(B_tmp)

3. Kandidat Matriks Sementara B (Terdapat 24 kemungkinan letak elemen dominan n):

Kandidat ke-1 dari sikel: (1)
[[  4. -inf -inf -inf]
 [-inf   4. -inf -inf]
 [-inf -inf   4. -inf]
 [-inf -inf -inf   4.]]

Kandidat ke-2 dari sikel: (1 2 3 4)
[[-inf   4. -inf -inf]
 [-inf -inf   4. -inf]
 [-inf -inf -inf   4.]
 [  4. -inf -inf -inf]]

Kandidat ke-3 dari sikel: (1 3)(2 4)
[[-inf -inf   4. -inf]
 [-inf -inf -inf   4.]
 [  4. -inf -inf -inf]
 [-inf   4. -inf -inf]]

Kandidat ke-4 dari sikel: (1 4)
[[-inf -inf -inf   4.]
 [-inf   4. -inf -inf]
 [-inf -inf   4. -inf]
 [  4. -inf -inf -inf]]

Kandidat ke-5 dari sikel: (2 3 4)
[[  4. -inf -inf -inf]
 [-inf -inf   4. -inf]
 [-inf -inf -inf   4.]
 [-inf   4. -inf -inf]]

Kandidat ke-6 dari sikel: (1 2 4 3)
[[-inf   4. -inf -inf]
 [-inf -inf -inf   4.]
 [  4. -inf -inf -inf]
 [-inf -inf   4. -inf]]

Kandidat ke-7 dari sikel: (1 3 2)
[[-inf -inf   4. -inf]
 [  4. -inf -inf -inf]
 [-inf   4. -inf -inf]
 [-inf -inf -inf   4.]]

Kandidat ke-8 dari s

## Langkah 4: Mencari Permutasi Elemen Terbesar Kedua

### Landasan Teori (Dekomposisi Latin Square)
Secara kombinatorik, sebuah matriks Latin Square berorde $n$ sebenarnya terbentuk dari $n$ buah pemetaan permutasi (disebut juga *transversal*) yang **saling lepas** (*disjoint*).
Artinya, setiap angka $\{1, 2, ..., n\}$ akan menempati posisi permutasinya masing-masing dengan syarat antar angka posisinya **tidak tumpang tindih**. 

Jika letak elemen terbesar $n$ pada matriks $B$ dipresentasikan oleh permutasi $\sigma$, dan elemen terbesar kedua $n-1$ kita letakkan berdasarkan permutasi $\tau$, maka matriks tidak akan bertabrakan jika dan hanya jika:
$$\sigma(i) \neq \tau(i) \quad \forall i \in \{1, 2, ..., n\}$$
Peristiwa di mana $\tau$ tidak memiliki elemen dengan letak yang sama dengan $\sigma$ sering dikaitkan dengan istilah kekacauan letak (*derangement*) relatif terhadap $\sigma$.

Oleh karena itu, untuk melanjutkan pencarian elemen kedua, kita iterasi pada **semua matriks kandidat $B$ sementara**, dan mencari himpunan permutasi yang *disjoint* dengan elemen dominan mereka, guna mempertahankan kodratnya menuju matriks Latin Square.

In [5]:
import itertools

def get_disjoint_permutations(sigma, n):
    """
    Mencari semua permutasi tau (derangement relatif) yang saling lepas 
    (disjoint/tidak tumpang tindih) dengan pemetaan sigma.
    """
    valid_perms = []
    # Gunakan 0-based index range untuk simulasi iterasi
    for p in itertools.permutations(range(n)):
        # Periksa apakah ada titik tetap (tumpang-tindih koordinat barisnya p(i) == sigma(i))
        if all(p[i] != sigma(i) for i in range(n)):
            valid_perms.append(Permutation(list(p)))
    return valid_perms

def add_second_largest_element(B_temp, tau, n):
    """
    Menambahkan elemen (n - 1) ke posisi yang ditunjuk oleh permutasi tau.
    """
    B_new = B_temp.copy()
    val = float(n - 1)  # Harus float karena B_temp masih mengandung float -inc (-np.inf)
    for i in range(n):
        # Set j ke letak elemen untuk baris i sesuai dengan pemetaan permutasi tau
        j = tau(i)
        B_new[i, j] = val
    return B_new

# ================================
# Contoh memproses kandidat Matriks B dengan elemen dominan n-1
print("4. Merekam Permutasi untuk Elemen Terbesar Kedua (n-1):")

candidates_step_4 = []
total_B_star_step_4 = 0

# Iterasi pada setiap matriks B kandidat yang memiliki elemen n (dari himpunan centralizer)
for idx, (sigma, B_tmp) in enumerate(zip(center_pi, B_temps)):
    disjoint_taus = get_disjoint_permutations(sigma, n_size)
    total_B_star_step_4 += len(disjoint_taus)
    candidates_step_4.append({
        'sigma': sigma,
        'taus': disjoint_taus,
        'base_B': B_tmp
    })
    
    sigma_str = print_cycle_1_based(sigma, n_size)
    print(f"\n[{idx + 1}] Untuk kandidat B dengan elemen n di sigma = {sigma_str}")
    print(f"    Ditemukan {len(disjoint_taus)} kemungkinan letak elemen {n_size - 1} (tau disjoint):")
    
    for j, tau in enumerate(disjoint_taus):
        tau_str = print_cycle_1_based(tau, n_size)
        print(f"      - Alternatif #{j + 1}: Sikel tau = {tau_str}")
        
    if disjoint_taus:
        # Demonstrasi hasil jika kita pilih alternatif tau pertama
        B_new = add_second_largest_element(B_tmp, disjoint_taus[0], n_size)
        print(f"        -> Contoh matriks jika kita gunakan Alternatif #1 untuk B ini:")
        # Indentasi cetakan NumPy agar selaras dengan output
        B_new_str = str(B_new).replace('\n', '\n           ')
        print(f"           {B_new_str}")

print("=" * 60)
print(f"Berdasarkan Langkah 4 ini (n dan n-1), kita memiliki Total {total_B_star_step_4} matriks B* sementara (bercirikan 2 elemen telah terisi).")

4. Merekam Permutasi untuk Elemen Terbesar Kedua (n-1):

[1] Untuk kandidat B dengan elemen n di sigma = (1)
    Ditemukan 9 kemungkinan letak elemen 3 (tau disjoint):
      - Alternatif #1: Sikel tau = (1 2)(3 4)
      - Alternatif #2: Sikel tau = (1 2 3 4)
      - Alternatif #3: Sikel tau = (1 2 4 3)
      - Alternatif #4: Sikel tau = (1 3 4 2)
      - Alternatif #5: Sikel tau = (1 3)(2 4)
      - Alternatif #6: Sikel tau = (1 3 2 4)
      - Alternatif #7: Sikel tau = (1 4 3 2)
      - Alternatif #8: Sikel tau = (1 4 2 3)
      - Alternatif #9: Sikel tau = (1 4)(2 3)
        -> Contoh matriks jika kita gunakan Alternatif #1 untuk B ini:
           [[  4.   3. -inf -inf]
            [  3.   4. -inf -inf]
            [-inf -inf   4.   3.]
            [-inf -inf   3.   4.]]

[2] Untuk kandidat B dengan elemen n di sigma = (1 2 3 4)
    Ditemukan 9 kemungkinan letak elemen 3 (tau disjoint):
      - Alternatif #1: Sikel tau = (1)
      - Alternatif #2: Sikel tau = (2 4 3)
      - Alternat

## Langkah 5: Penyelesaian dengan Sistem Persamaan Linear (SPL) Max-Plus

Alih-alih melakukan tebak-kombinasi (*generate and test*) untuk sisa elemen yang kosong di matriks $B^*$, kita dapat **mengkonstruksi secara matematis sisa mutlaknya** menggunakan prinsip penyelesaian **Sistem Persamaan Linear (SPL) Max-Plus**. 

Ingat kembali persamaan komutatif yang ingin dicapai:
$$ A \otimes B = B \otimes A $$

Jika kita memandang persamaan ini dalam lingkup vektor kolom, maka untuk setiap kolom ke-$j$ pada matriks $B$, berlaku:
$$ A \otimes b_j = y_j $$
dimana $b_j$ adalah vektor kolom ke-$j$ dari matriks $B$, dan $y_j$ adalah vektor kolom hasil dari sisi kanan $(B \otimes A)_j$.

Karena pada Langkah 4 matriks temporer $B^*$ telah memiliki elemen penguasa (yaitu angka $n$ dan $n-1$), elemen nilai maksimum yang sudah kita tempatkan di $B^*$ ini **sangat mendominasi** hasil dari perkalian Max-Plus. Sehingga, vektor target $y_j = (B^* \otimes A)_j$ nilainya mutlak tidak akan berubah meski ada sel rumpang ($-\infty$) di $B^*$.

### Algoritma Pemecahan Terurut (Step-by-Step SPL Max-Plus):
Setelah kita mendapatkan vektor target mutlak $y_j$, tugas kita adalah mancari angka untuk sel rumpang pada $b_j$. Hal ini direduksi menjadi penyelesaian SPL $A \otimes x = y_j$.

**Tahap 1: Membangun Matriks Ketidaksesuaian ($D$)**
Untuk mencari batas toleransi angka yang bisa dimasukkan, kita buat Matriks Ketidaksesuaian $D$. Secara definisi, matriks ini dibentuk dari pengurangan nilai target $y_j$ dengan setiap elemen matriks koefisien $A$:
$$ D_{i,k} = y_i - a_{i,k} $$

**Tahap 2: Menentukan Subsolusi Terbesar ($\bar{x}$)**
Dalam Aljabar Max-Plus, solusi maksimum (batas atas) dari SPL tersebut, dinotasikan $\bar{x}$, didapatkan dengan mencari nilai minimum dari setiap **kolom** matriks $D$:
$$ \bar{x}_k = \min_{i} \{ D_{i,k} \} $$
Elemen $\bar{x}_k$ inilah yang menjadi "Lampu Hijau" atau kapasitas maksimum angka yang **boleh diisikan** pada baris ke-$k$ (atau slot kosong di $B^*$) tanpa merusak tatanan hasil maksimum $(B \otimes A)$.

**Tahap 3: Matriks Tereduksi ($R$) dan Alokasi**
Untuk melihat keketatan/struktur kemajemukan solusi, kita petakan $R_{i,k} = 1$ jika $D_{i,k} = \bar{x}_k$ (dan $0$ jika tidak). Di dalam blok algoritma komputasi di bawah:
- Sel kosong pada $B^*$ akan dikumpulkan.
- Tersedia opsi sisa elemen Latin Square (misal angka $2$ dan $1$).
- Kita menggunakan pendekatan **Greedy Konstruktif**: memasangkan susunan elemen sisa ini kepada slot kosong yang batas atasnya ($\bar{x}$) paling memadai! 
- Slot yang berkapasitas daya tampung batas SPL lebih besar, berhak mendapat prioritas diisi sisa elemen Latin Square yang angkanya lebih besar pula.

Karena ini adalah operasi Aljabar pemetaan murni per baris, maka dalam sekali jalan (deterministik $O(n^2)$), kita langsung memperoleh matriks utuh $B_{full}$ yang komutatif tanpa perlu perulangan bruteforce *(backtracking)*!

---
### Contoh Numerik Penentuan Elemen Menggunakan SPL Max-Plus (Orde $n=4$)
Misalkan untuk $n=4$, kita memiliki matriks Latin Square $A$:
$$ 
A = 
\begin{bmatrix}
4 & 3 & 2 & 1 \\
1 & 4 & 3 & 2 \\
2 & 1 & 4 & 3 \\
3 & 2 & 1 & 4
\end{bmatrix}
$$
Lalu pada Langkah 4, kita mendapatkan sebuah sub-kandidat $B^*$ (di mana sel kosong direpresentasikan dengan simbol $\cdot$ batas $-\infty$). Kita tinjau untuk **kolom pertama** (indeks $j=0$):
* Vektor kolom $b_0$ di bentuk asalnya: $\begin{bmatrix} \cdot & 4 & 3 & \cdot  \end{bmatrix}^T$
* Misal sisa elemen yang harus diletakkan adalah angka $\{2, 1\}$.
* Berdasarkan dominasi nilai $n=4$ dan $3$, target dari $(B^* \otimes A)$ untuk kolom nol sudah bersifat **final/fixed**. Misalkan didapatkan target:  
$$ y_0 = \begin{bmatrix} 5 \\ 8 \\ 7 \\ 6 \end{bmatrix} $$

Lalu kita menyelesaikan SPL: $A \otimes x = y_0$.

**1. Matriks Ketidaksesuaian $D$:**
Kurangkan target $y_0$ dengan setiap elemen di $A$ baris per baris ($D_{i,k} = y_{0,i} - a_{i,k}$):
$$
D = 
\begin{bmatrix}
5-4 & 5-3 & 5-2 & 5-1 \\
8-1 & 8-4 & 8-3 & 8-2 \\
7-2 & 7-1 & 7-4 & 7-3 \\
6-3 & 6-2 & 6-1 & 6-4
\end{bmatrix}
=
\begin{bmatrix}
1 & 2 & 3 & 4 \\
7 & 4 & 5 & 6 \\
5 & 6 & 3 & 4 \\
3 & 4 & 5 & 2
\end{bmatrix}
$$

**2. Subsolusi Terbesar ($\bar{x}$):**
Cari nilai minimum dari masing-masing *kolom* matriks $D$:
* Kolom 1: $\min(1, 7, 5, 3) = 1$
* Kolom 2: $\min(2, 4, 6, 4) = 2$  *(Angka 4 yang sudah ada, dibatasi batas ideal <= 2. Tunggu, contoh ilustrasi ini fiktif untuk menunjukkan metode)*
* Kolom 3: $\min(3, 5, 3, 5) = 3$  *(Angka 3 yang sudah ada masuk)*
* Kolom 4: $\min(4, 6, 4, 2) = 2$

Diperoleh $\bar{x} = \begin{bmatrix} 1 & 2 & 3 & 2 \end{bmatrix}^T$.
*(Catatan: Ini berarti pada indeks 0 kapasitas maksimumnya adalah 1, dan pada indeks 3 kapasitas maksimumnya adalah 2).*

**3. Pemetaan Konstruktif (Alokasi ke Sel Kosong):**
Elemen nol-indikatif tempat kosong (indeks ke-$0$ dan ke-$3$) difilter:
* Kosong di indeks $0$: batas $\bar{x}[0] = 1$
* Kosong di indeks $3$: batas $\bar{x}[3] = 2$

Angka Latin Square yang tersisa untuk diisikan adalah $\{2, 1\}$. 
Berdasarkan pendekatan **Greedy/Batas Maksimum**, sisa angka terbesar (yaitu $2$) akan diisikan ke slot yang batas tampungnya ($\bar{x}$) lebih besar, yakni indeks $3$. Sedangkan sisa angka terkecil ($1$), dialokasikan ke indeks $0$.
* Indeks $3 \leftarrow$ elemen sisa $2$
* Indeks $0 \leftarrow$ elemen sisa $1$

Sehingga, dari yang awalnya rumpang $\begin{bmatrix} \cdot & 4 & 3 & \cdot  \end{bmatrix}^T$, tersusun utuh secara mutlak menjadi $\begin{bmatrix} 1 & 4 & 3 & 2 \end{bmatrix}^T$. 

Inilah bagaimana **Sistem Persamaan Linear Max-Plus menyelesaikan rumpang matriks persis secara logis, bukan tebak-tebakan.**

In [6]:
import numpy as np

def max_plus_mul(X, Y):
    """ Perkalian Max-Plus standard. """
    n = X.shape[0]
    Z = np.full((n, n), -np.inf)
    for i in range(n):
        for j in range(n):
            Z[i, j] = np.max(X[i, :] + Y[:, j])
    return Z

def min_plus_mul(X, Y):
    """ Perkalian Min-Plus dual untuk mencari Subsolusi. """
    n = X.shape[0]
    Z = np.full((n, n), np.inf)
    for i in range(n):
        for j in range(n):
            Z[i, j] = np.min(X[i, :] + Y[:, j])
    return Z

def is_latin_square(M, n):
    """ Validasi ketat kelayakan konstruksi Latin Square """
    for i in range(n):
        if set(M[i, :]) != set(range(1, n + 1)): return False
        if set(M[:, i]) != set(range(1, n + 1)): return False
    return True

def apply_SPL_construction(A, B_star, n):
    """
    Mengubah pencarian sisanya menggunakan teknik penyelesaian eksak SPL Max-Plus!
    """
    # 1. Hitung Target Vektor (Y) dari (B* ⊗ A)
    # Karena B* punya elemen dominan n dan n-1, max-plus akan langsung mematok batas ketat!
    Y = max_plus_mul(B_star, A)
    
    # 2. Hitung Subsolusi Terbesar (B_bar = -A^T ⊗' Y)
    # Ini merupakan 'Discrepancy Matrix' boundary untuk tiap sel yang rumpang (-inf)
    minus_A_T = -A.T
    B_bar = min_plus_mul(minus_A_T, Y)
    
    # 3. Alokasi Heuristik/Konstruktif berdasarkan panduan B_bar
    B_full = B_star.copy()
    
    # Pendekatan Greedy: Di tiap baris, pasangkan angka terbesar yang sisa
    # ke kolom rumpang dengan batas subsolution (B_bar) terbesar!
    for i in range(n):
        existing_vals = set(B_full[i, :])
        missing_vals = sorted(list(set(range(1, n + 1)) - existing_vals), reverse=True)
        
        empty_cols = [j for j in range(n) if B_full[i, j] == -np.inf]
        
        # Urutkan berdasarkan kapasitas maksimum B_bar menurun
        empty_cols_sorted = sorted(empty_cols, key=lambda j: B_bar[i, j], reverse=True)
        
        for idx, j in enumerate(empty_cols_sorted):
            val = missing_vals[idx]
            B_full[i, j] = float(val)
            
    return B_full

print(f"5. Penyelesaian Konstruktif SPL Max-Plus (A ⊗ B == B ⊗ A) untuk n = {n_size}\n")
valid_candidates_step_5 = []
total_b_full_candidates = 0

for idx, cand in enumerate(candidates_step_4):
    sigma = cand['sigma']
    taus = cand['taus']
    base_B = cand['base_B']
    
    for j, tau in enumerate(taus):
        B_star = add_second_largest_element(base_B, tau, n_size)
        
        # Eksekusi Vektor SPL Murni (Satu eksekusi deterministik berdasar limit, TANPA LOOP KEMUNGKINAN!!)
        B_full = apply_SPL_construction(A, B_star, n_size)
        total_b_full_candidates += 1
        
        # Karena kita menebak/mapping sisa sel secara linear greedy, 
        # kita hanya perlu memvalidasi apakah tabrakan ditiadakan (Valid LS) & Komutatif.
        if is_latin_square(B_full, n_size):
            AB = max_plus_mul(A, B_full)
            BA = max_plus_mul(B_full, A)
            
            if np.array_equal(AB, BA):
                # Filter redundansi di penambahan
                B_int = B_full.astype(int)
                if not any(np.array_equal(B_int, v['B_full']) for v in valid_candidates_step_5):
                    valid_candidates_step_5.append({
                        'sigma': sigma,
                        'tau': tau,
                        'B_full': B_int,
                        'AB': AB.astype(int),
                        'BA': BA.astype(int)
                    })

def format_matrix(M):
    return str(M).replace('\n', '\n          ')

print("=" * 60)
print(f"Total matriks B* hasil kombinasi deterministik vektor B_bar (TIDAK ADA BACKTRACK REKURSI!): {total_b_full_candidates} matriks.")
print(f"DITEMUKAN {len(valid_candidates_step_5)} matriks unik B yang KOMUTATIF SEMPURNA hasil rekonstruksi dari subsolusi:\n")

for i, valid in enumerate(valid_candidates_step_5):
    sigma_str = print_cycle_1_based(valid['sigma'], n_size)
    tau_str = print_cycle_1_based(valid['tau'], n_size)
    print("=" * 60)
    print(f"Solusi Matriks B Ke-#{i + 1}")
    print(f" > Konstruksi Dominan   (n={n_size}, n-1={n_size-1})")
    print(f" > Konstruksi SPL Baris (Memasangkan angka dari subsolusi B_bar)")
    
    print("\n   Matriks L.S. A:")
    print(f"          {format_matrix(A)}\n")
    print("   Matriks komutatif B hasil SPL:")
    print(f"          {format_matrix(valid['B_full'])}\n")

5. Penyelesaian Konstruktif SPL Max-Plus (A ⊗ B == B ⊗ A) untuk n = 4

Total matriks B* hasil kombinasi deterministik vektor B_bar (TIDAK ADA BACKTRACK REKURSI!): 216 matriks.
DITEMUKAN 36 matriks unik B yang KOMUTATIF SEMPURNA hasil rekonstruksi dari subsolusi:

Solusi Matriks B Ke-#1
 > Konstruksi Dominan   (n=4, n-1=3)
 > Konstruksi SPL Baris (Memasangkan angka dari subsolusi B_bar)

   Matriks L.S. A:
          [[4 2 3 1]
           [1 4 2 3]
           [3 1 4 2]
           [2 3 1 4]]

   Matriks komutatif B hasil SPL:
          [[4 3 2 1]
           [3 4 1 2]
           [2 1 4 3]
           [1 2 3 4]]

Solusi Matriks B Ke-#2
 > Konstruksi Dominan   (n=4, n-1=3)
 > Konstruksi SPL Baris (Memasangkan angka dari subsolusi B_bar)

   Matriks L.S. A:
          [[4 2 3 1]
           [1 4 2 3]
           [3 1 4 2]
           [2 3 1 4]]

   Matriks komutatif B hasil SPL:
          [[4 3 2 1]
           [1 4 3 2]
           [2 1 4 3]
           [3 2 1 4]]

Solusi Matriks B Ke-#3
 > Konstruk

In [7]:
print(f"Berdasarkan algoritma SPL Heuristic, didapat jumlah matriks: {len(valid_candidates_step_5)}")

Berdasarkan algoritma SPL Heuristic, didapat jumlah matriks: 36
